In [54]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import skew
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.utils import resample
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [55]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [56]:
root_dir = 'C:/Users/MELİSA/smaller_dataset.csv'
df = pd.read_csv(root_dir)

In [57]:
df["Weight"] = df["Total Fwd Packet"] * df["Total Bwd packets"]

In [ ]:

rename_mapping = {
    "Benign&Bruteforce_benign": "Benign",
    "Benign&Bruteforce_BruteForce": "Brute Force"
}


df['Attack_Type'] = df['Attack_Type'].replace(rename_mapping)


In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]  

for col in categorical_columns:
    df[col] = df[col].astype('category')

In [60]:
for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [61]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [62]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Benign&Bruteforce_BruteForce",
    "Uploading_Attack"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [63]:
columns_to_drop = ["Fwd Packet Length Mean",
                   "Fwd Packet Length Min",
                   "Fwd Packet Length Std",
                   "Fwd Segment Size Avg",
                   "Total Bwd packets",
                   "Bwd Segment Size Avg",
                   "Bwd PSH Flags", 
                   "Fwd URG Flags", 
                   "Fwd IAT Min",
                   "Bwd URG Flags",
                   "Bwd Packet Length Mean",
                   "Bwd Packet Length Std",
                   "Flow IAT Min",	
                   "Flow IAT Max",
                   "Flow IAT Mean",	
                   "Flow IAT Max",    
                   "Fwd IAT Std",	
                   "Fwd IAT Max",
                   "CWR Flag Count",      
                   "Bwd IAT Min",
                   "Bwd IAT Max",
                   "Packet Length Mean",
                   "Packet Length Std",
                   "Packet Length Variance",
                   "Fwd Bytes/Bulk Avg",
                   "Fwd Packet/Bulk Avg",
                   "Fwd Bulk Rate Avg",
                   "Active Max",
                   "Active Min",
                   "Idle Max",
                   "Idle Min",
                   "Idle Std",
                   "Idle Mean",
                   "Subflow Fwd Packets",
                   "Fwd Header Length",
                   "Fwd IAT Total",
                   "Total Length of Fwd Packet",
                   "Fwd Act Data Pkts",
                   "Fwd Packet Length Max",
                   "Fwd Packets/s",
                   "Bwd Bytes/Bulk Avg",
                   "Bwd Packet/Bulk Avg",
                   "Bwd Header Length",
                   "Flow ID",
                   "URG Flag Count",
                   "Label"]

df = df.drop(columns=columns_to_drop)

In [64]:
print("Shape of the dataset:", df.shape)

print("\nColumns with Null Values:")
null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]  
print(null_columns)

print("\nColumns with Inf Values:")
inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
print(inf_columns)

df = df.dropna()
print("Shape of the dataset after dropping Null:", df.shape)

df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
print("Shape of the dataset after dropping Inf:", df.shape)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
print("Infs converted to NaNs.")
print(df.isnull().sum()[df.isnull().sum() > 0])



Shape of the dataset: (439194, 47)

Columns with Null Values:
Flow Bytes/s    170
dtype: int64

Columns with Inf Values:
Index(['Flow Bytes/s', 'Flow Packets/s'], dtype='object')
Shape of the dataset after dropping Null: (439024, 47)
Shape of the dataset after dropping Inf: (438920, 47)
Infs converted to NaNs.
Series([], dtype: int64)


In [ ]:

non_numeric_df = df.select_dtypes(exclude=[np.number])


non_numeric_df.head()



,Src IP,Src Port,Dst IP,Dst Port,Timestamp,Attack_Type,Proto_0,Proto_6,Proto_17
0,192.168.137.144,49153,192.168.137.240,13217,2022-08-05 10:53:38,DoS_DoS SYN Flood,False,True,False
1,192.168.137.65,7661,192.168.137.171,6668,2022-10-26 11:53:56,DDoS_DDoS ACK Fragmentation,False,True,False
2,192.168.137.12,15376,192.168.137.235,8008,2022-10-26 15:46:03,DDoS_DDoS ACK Fragmentation,False,True,False
3,192.168.137.12,21499,192.168.137.206,55442,2022-10-26 16:50:41,DDoS_DDoS ACK Fragmentation,False,True,False
4,192.168.137.225,38616,192.168.137.132,8080,2022-09-14 11:10:19,DDoS_DDoS-HTTP Flood,False,True,False


In [66]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign':  
        return 'Normal'
    else:
        return 'Attack'

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

print(df[['Attack_Type', 'Anomaly_Label']].head())

                   Attack_Type Anomaly_Label
0            DoS_DoS SYN Flood        Attack
1  DDoS_DDoS ACK Fragmentation        Attack
2  DDoS_DDoS ACK Fragmentation        Attack
3  DDoS_DDoS ACK Fragmentation        Attack
4         DDoS_DDoS-HTTP Flood        Attack


In [67]:
df.head()

,Src IP,Src Port,Dst IP,Dst Port,Timestamp,Flow Duration,Total Fwd Packet,Total Length of Bwd Packet,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,Flow Packets/s,Flow IAT Std,Fwd IAT Mean,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Fwd PSH Flags,Bwd Packets/s,Packet Length Min,Packet Length Max,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Bwd Bulk Rate Avg,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Seg Size Min,Active Mean,Active Std,Attack_Type,Weight,Src IP_freq,Dst IP_freq,Src Port_freq,Dst Port_freq,Proto_0,Proto_6,Proto_17,Anomaly_Label
0,192.168.137.144,49153,192.168.137.240,13217,2022-08-05 10:53:38,18188538,1,0.0,0.0,0.0,0.000000,0.274898,2.655540e+06,0.0,13184858.0,4.394953e+06,3.230926e+06,0,0.219919,0.0,0.0,0,4,1,0,1,0,4.0,0.0,0,0,1,0,29200,0,24,0.0,0.0,DoS_DoS SYN Flood,4,1361,1363,1417,1,False,True,False,Attack
1,192.168.137.65,7661,192.168.137.171,6668,2022-10-26 11:53:56,99736150,2,0.0,0.0,0.0,29.277248,0.020053,0.000000e+00,99736150.0,0.0,0.000000e+00,0.000000e+00,0,0.000000,1460.0,1460.0,0,0,0,0,2,0,0.0,2190.0,0,2920,0,0,512,0,20,0.0,0.0,DDoS_DDoS ACK Fragmentation,0,658,116,1,9306,False,True,False,Attack
2,192.168.137.12,15376,192.168.137.235,8008,2022-10-26 15:46:03,88834040,2,0.0,0.0,0.0,32.870283,0.022514,0.000000e+00,88834040.0,0.0,0.000000e+00,0.000000e+00,0,0.000000,1460.0,1460.0,0,0,0,0,2,0,0.0,2190.0,0,2920,0,0,512,0,20,0.0,0.0,DDoS_DDoS ACK Fragmentation,0,283,777,2,960,False,True,False,Attack
3,192.168.137.12,21499,192.168.137.206,55442,2022-10-26 16:50:41,17797062,1,0.0,0.0,0.0,82.036012,0.112378,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0,0.056189,0.0,1460.0,0,0,1,0,1,0,1.0,1460.0,0,1460,1,0,512,0,20,0.0,0.0,DDoS_DDoS ACK Fragmentation,1,283,131,1,702,False,True,False,Attack
4,192.168.137.225,38616,192.168.137.132,8080,2022-09-14 11:10:19,294063,1,0.0,0.0,0.0,0.000000,6.801264,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0,3.400632,0.0,0.0,0,1,1,0,1,0,1.0,0.0,0,0,0,0,64240,0,40,0.0,0.0,DDoS_DDoS-HTTP Flood,1,57,336,5,2296,False,True,False,Attack


In [68]:
X = df.select_dtypes(include=['number', 'bool'])
y = df['Anomaly_Label']  # Target variable
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [69]:
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif


anova_selector = SelectKBest(score_func=f_classif, k=33)
X_anova = anova_selector.fit_transform(X, y)
anova_scores = anova_selector.scores_


mi_selector = SelectKBest(score_func=mutual_info_classif, k=33)
X_mi = mi_selector.fit_transform(X, y)
mi_scores = mi_selector.scores_


In [ ]:
anova_df = pd.DataFrame({
    'Feature': X.columns,
    'ANOVA_F_score': anova_scores
}).sort_values(by='ANOVA_F_score', ascending=False)


mi_df = pd.DataFrame({
    'Feature': X.columns,
    'Mutual_Information': mi_scores
}).sort_values(by='Mutual_Information', ascending=False)


In [72]:
anova_df

,Feature,ANOVA_F_score
17,SYN Flag Count,239655.488814
18,RST Flag Count,43497.667253
31,Active Mean,41808.744339
32,Active Std,34164.774626
37,Dst Port_freq,33224.811781
39,Proto_6,29117.444601
40,Proto_17,25959.165721
0,Flow Duration,23234.266344
35,Dst IP_freq,21654.593448
34,Src IP_freq,17992.821358


In [73]:
mi_df

,Feature,Mutual_Information
34,Src IP_freq,0.275291
35,Dst IP_freq,0.266437
37,Dst Port_freq,0.204484
15,Packet Length Max,0.180981
23,Average Packet Size,0.180772
28,FWD Init Win Bytes,0.174762
5,Flow Bytes/s,0.140552
6,Flow Packets/s,0.139263
17,SYN Flag Count,0.135144
8,Fwd IAT Mean,0.131156


In [74]:
selected_features_anova = X.columns[anova_selector.get_support()]

In [75]:
selected_features_anova

Index(['Flow Duration', 'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Std', 'Fwd IAT Mean',
       'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Fwd PSH Flags',
       'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max',
       'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'ECE Flag Count', 'Average Packet Size',
       'FWD Init Win Bytes', 'Bwd Init Win Bytes', 'Fwd Seg Size Min',
       'Active Mean', 'Active Std', 'Src IP_freq', 'Dst IP_freq',
       'Src Port_freq', 'Dst Port_freq', 'Proto_0', 'Proto_6', 'Proto_17'],
      dtype='object')

In [76]:
selected_features_mi = X.columns[mi_selector.get_support()]

In [77]:
selected_features_mi 

Index(['Flow Duration', 'Total Fwd Packet', 'Total Length of Bwd Packet',
       'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Std', 'Fwd IAT Mean', 'Bwd IAT Total',
       'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd Packets/s', 'Packet Length Min',
       'Packet Length Max', 'SYN Flag Count', 'RST Flag Count',
       'ACK Flag Count', 'Down/Up Ratio', 'Average Packet Size',
       'Subflow Fwd Bytes', 'FWD Init Win Bytes', 'Bwd Init Win Bytes',
       'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Weight',
       'Src IP_freq', 'Dst IP_freq', 'Src Port_freq', 'Dst Port_freq',
       'Proto_6', 'Proto_17'],
      dtype='object')

In [78]:
y_pred = rf_classifier.predict(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd

lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),  
    ('lasso', LogisticRegression(penalty='l1', solver='liblinear', C=0.1))  
])

lasso_pipeline.fit(X, y)
lasso_coef = lasso_pipeline.named_steps['lasso'].coef_[0]

lasso_df = pd.DataFrame({
    'Feature': X.columns,
    'L1_Coefficient': lasso_coef
}).sort_values(by='L1_Coefficient', key=abs, ascending=False)


selected_lasso_features = lasso_df[lasso_df['L1_Coefficient'] != 0]

C:\Users\MELİSA\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [82]:
selected_lasso_features

,Feature,L1_Coefficient
1,Total Fwd Packet,-4.344087
17,SYN Flag Count,-2.826579
19,PSH Flag Count,2.775195
33,Weight,1.854991
22,Down/Up Ratio,-1.394881
30,Fwd Seg Size Min,1.268728
15,Packet Length Max,1.245270
9,Bwd IAT Total,1.180988
3,Bwd Packet Length Max,1.064688
35,Dst IP_freq,1.008597


In [81]:
from sklearn.ensemble import RandomForestClassifier

rf_importances = rf_classifier.feature_importances_

rf_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_importances
}).sort_values(by='Importance', ascending=False)


In [83]:
rf_df

,Feature,Importance
17,SYN Flag Count,0.159775
36,Src Port_freq,0.072464
35,Dst IP_freq,0.059235
23,Average Packet Size,0.057244
5,Flow Bytes/s,0.057001
15,Packet Length Max,0.051302
37,Dst Port_freq,0.043230
6,Flow Packets/s,0.041001
28,FWD Init Win Bytes,0.038060
34,Src IP_freq,0.036237
